In [1]:
import json

with open(
    "../data/processed/papers_merged.json",
    "r",
    encoding="utf-8"
) as f:

    papers = json.load(f)

print("Total papers:", len(papers))

Total papers: 179


In [2]:
from collections import Counter

label_counts = Counter()

for paper in papers:

    for label in paper["labels"]:
        label_counts[label] += 1


print("\n" + "=" * 60)
print("LABEL COUNTS")
print("=" * 60)

for label, count in sorted(label_counts.items()):

    print(f"{label}: {count}")


LABEL COUNTS
Computer Vision: 50
Machine Learning: 50
NLP: 50
Robotics: 50


In [3]:
# ============================================================
# LABEL COMBINATION DISTRIBUTION
# ============================================================

combination_counts = Counter()

for paper in papers:

    combination = " + ".join(
        sorted(paper["labels"])
    )

    combination_counts[combination] += 1


print("\n" + "=" * 60)
print("LABEL COMBINATION DISTRIBUTION")
print("=" * 60)

for combination, count in sorted(
    combination_counts.items()
):

    print(f"{combination}: {count}")


LABEL COMBINATION DISTRIBUTION
Computer Vision: 36
Computer Vision + Machine Learning: 10
Computer Vision + Machine Learning + Robotics: 1
Computer Vision + NLP: 1
Computer Vision + Robotics: 2
Machine Learning: 33
Machine Learning + NLP: 5
Machine Learning + Robotics: 1
NLP: 44
Robotics: 46


Label Encode

In [4]:


label_names = [
    "NLP",
    "Computer Vision",
    "Machine Learning",
    "Robotics"
]

label_to_index = {
    label: index
    for index, label in enumerate(label_names)
}

print(label_to_index)

{'NLP': 0, 'Computer Vision': 1, 'Machine Learning': 2, 'Robotics': 3}


In [5]:
import numpy as np

Y = np.zeros(
    (len(papers), len(label_names)),
    dtype=int
)

for i, paper in enumerate(papers):

    for label in paper["labels"]:

        label_index = label_to_index[label]

        Y[i, label_index] = 1

In [6]:
print("Y shape:", Y.shape)

Y shape: (179, 4)


In [7]:
for i in range(10):

    print(
        papers[i]["title"]
    )

    print(
        papers[i]["labels"]
    )

    print(
        Y[i]
    )

    print("-" * 60)

End-to-End Speaker Diarization as Post-Processing
['NLP']
[1 0 0 0]
------------------------------------------------------------
Regularized Attentive Capsule Network for Overlapped Relation Extraction
['NLP']
[1 0 0 0]
------------------------------------------------------------
Should I visit this place? Inclusion and Exclusion Phrase Mining from Reviews
['NLP']
[1 0 0 0]
------------------------------------------------------------
Speech Synthesis as Augmentation for Low-Resource ASR
['NLP']
[1 0 0 0]
------------------------------------------------------------
QUACKIE: A NLP Classification Task With Ground Truth Explanations
['NLP']
[1 0 0 0]
------------------------------------------------------------
I like fish, especially dolphins: Addressing Contradictions in Dialogue Modeling
['NLP', 'Machine Learning']
[1 0 1 0]
------------------------------------------------------------
Panarchy: ripples of a boundary concept
['NLP']
[1 0 0 0]
----------------------------------------------

2. Train / Validation / Test split

In [8]:
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

X = np.arange(len(papers))

# ------------------------------------------------------------
# STEP 1: %70 Train, %30 Temporary
# ------------------------------------------------------------

msss = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=0.30,
    random_state=42
)

train_idx, temp_idx = next(
    msss.split(X, Y)
)

print("Train:", len(train_idx))
print("Temporary:", len(temp_idx))

Train: 124
Temporary: 55


In [9]:
Y_temp = Y[temp_idx]

print("Temporary papers:", len(temp_idx))
print("Y_temp shape:", Y_temp.shape)

Temporary papers: 55
Y_temp shape: (55, 4)


In [10]:
X_temp = np.arange(len(temp_idx))

print(X_temp)

[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47
 48 49 50 51 52 53 54]


In [11]:
msss_temp = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=0.50,
    random_state=42
)

In [12]:
val_relative_idx, test_relative_idx = next(
    msss_temp.split(X_temp, Y_temp)
)

In [13]:
val_idx = temp_idx[val_relative_idx]
test_idx = temp_idx[test_relative_idx]

In [14]:
print("=" * 60)
print("FINAL SPLIT")
print("=" * 60)

print("Train:", len(train_idx))
print("Validation:", len(val_idx))
print("Test:", len(test_idx))

print("\nTotal:", len(train_idx) + len(val_idx) + len(test_idx))

FINAL SPLIT
Train: 124
Validation: 26
Test: 29

Total: 179


Leakeg control

In [15]:
train_set = set(train_idx)
val_set = set(val_idx)
test_set = set(test_idx)

print("=" * 60)
print("SPLIT OVERLAP CHECK")
print("=" * 60)

print("Train ∩ Validation:", len(train_set & val_set))
print("Train ∩ Test:", len(train_set & test_set))
print("Validation ∩ Test:", len(val_set & test_set))

SPLIT OVERLAP CHECK
Train ∩ Validation: 0
Train ∩ Test: 0
Validation ∩ Test: 0


In [16]:
# ============================================================
# LABEL DISTRIBUTION FUNCTION
# ============================================================

def show_label_distribution(name, indices):

    counts = Y[indices].sum(axis=0)

    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)

    for label, count in zip(label_names, counts):
        print(f"{label}: {int(count)}")

In [17]:
show_label_distribution("ALL DATA", np.arange(len(papers)))

show_label_distribution("TRAIN", train_idx)

show_label_distribution("VALIDATION", val_idx)

show_label_distribution("TEST", test_idx)


ALL DATA
NLP: 50
Computer Vision: 50
Machine Learning: 50
Robotics: 50

TRAIN
NLP: 35
Computer Vision: 35
Machine Learning: 35
Robotics: 35

VALIDATION
NLP: 7
Computer Vision: 7
Machine Learning: 7
Robotics: 7

TEST
NLP: 8
Computer Vision: 8
Machine Learning: 8
Robotics: 8


In [18]:
# ============================================================
# CREATE TRAIN / VALIDATION / TEST DATASETS
# ============================================================

train_papers = [papers[i] for i in train_idx]

validation_papers = [papers[i] for i in val_idx]

test_papers = [papers[i] for i in test_idx]


print("Train papers:", len(train_papers))
print("Validation papers:", len(validation_papers))
print("Test papers:", len(test_papers))

Train papers: 124
Validation papers: 26
Test papers: 29


In [20]:
import os 
# ============================================================
# SAVE SPLITS
# ============================================================

processed_dir = "../data/processed"

os.makedirs(processed_dir, exist_ok=True)


with open(
    f"{processed_dir}/train_papers.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        train_papers,
        f,
        ensure_ascii=False,
        indent=2
    )


with open(
    f"{processed_dir}/validation_papers.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        validation_papers,
        f,
        ensure_ascii=False,
        indent=2
    )


with open(
    f"{processed_dir}/test_papers.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        test_papers,
        f,
        ensure_ascii=False,
        indent=2
    )


print("Datasets saved successfully.")

Datasets saved successfully.


In [21]:
with open(
    "../data/processed/train_papers.json",
    "r",
    encoding="utf-8"
) as f:

    train_check = json.load(f)


with open(
    "../data/processed/validation_papers.json",
    "r",
    encoding="utf-8"
) as f:

    validation_check = json.load(f)


with open(
    "../data/processed/test_papers.json",
    "r",
    encoding="utf-8"
) as f:

    test_check = json.load(f)


print("Train:", len(train_check))
print("Validation:", len(validation_check))
print("Test:", len(test_check))

Train: 124
Validation: 26
Test: 29


In [22]:
train_ids = {paper["id"] for paper in train_check}

validation_ids = {paper["id"] for paper in validation_check}

test_ids = {paper["id"] for paper in test_check}


print("Train ∩ Validation:",
      len(train_ids & validation_ids))

print("Train ∩ Test:",
      len(train_ids & test_ids))

print("Validation ∩ Test:",
      len(validation_ids & test_ids))

Train ∩ Validation: 0
Train ∩ Test: 0
Validation ∩ Test: 0
